# 암종 분류 프로젝트 · 01 데이터 전처리

**목적:** Train 데이터의 4,384개 유전자 변이 문자열을 유지하면서, 최초 Baseline 실험용 **WT=0 / 변이 존재=1** 특징을 만든다. 같은 **원본 유전체 프로파일**이 학습·검증에 동시에 나타나지 않도록 분할한다.

- **공식 지표:** Macro F1 (향후 모델 학습에서 사용)
- **데이터 누출 방지:** 이 노트북의 1차 전처리와 검증 분할은 **Train만 사용**한다. Test는 맨 아래의 별도 선택 셀에서 최종 추론 직전에만 변환한다.
- **주의:** WT(야생형), 실제 변이, 결측은 서로 다른 의미다. 현재 Train에는 결측이 없다는 전제하에 이진 기준선을 만든다. 결측이 생길 경우 이 노트북은 오류를 내고 멈추도록 설계했다.

**준비:** VS Code에서 Python 및 Jupyter 확장을 설치하고 이 파일을 연다. 같은 폴더의 `data` 폴더에 `train(1).csv`를 넣는다. `test(2).csv`는 지금 단계에서 필요하지 않다.

## 0. 패키지 설치 (최초 1회)
아래 셀을 실행한 뒤 필요하면 Notebook 커널을 재시작한다.

In [ ]:
%pip install pandas numpy scipy scikit-learn matplotlib

## 1. 경로 및 기본 설정

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 노트북이 있는 폴더에서 실행하는 것을 권장.
# data/ 안에 없는 경우 현재 폴더의 파일도 찾도록 지원.
def find_file(names):
    for base in [DATA_DIR, Path('.')]:
        for name in names:
            p = base / name
            if p.is_file():
                return p
    raise FileNotFoundError(f'파일을 찾을 수 없음: {names}. data/ 폴더를 확인하세요.')

print('현재 폴더:', Path.cwd().resolve())
print('결과 저장 폴더:', OUTPUT_DIR.resolve())

## 2. Train 불러오기 및 기본 검사

In [ ]:
train_path = find_file(['train(1).csv', 'train.csv'])
train = pd.read_csv(train_path, dtype='string', low_memory=False)
assert {'ID', 'SUBCLASS'}.issubset(train.columns), 'Train에 ID, SUBCLASS가 있어야 합니다.'
assert not train['ID'].isna().any(), 'ID 결측이 존재합니다.'
assert train['ID'].is_unique, 'ID 중복이 존재합니다.'
assert not train['SUBCLASS'].isna().any(), '정답 라벨 결측이 존재합니다.'

gene_columns = [c for c in train.columns if c not in ['ID', 'SUBCLASS']]
assert len(gene_columns) == len(set(gene_columns)), '유전자 열 이름 중복'
missing_train = int(train[gene_columns].isna().sum().sum())
print(f'Train: {len(train):,}명 | 유전자: {len(gene_columns):,}개 | 암종: {train.SUBCLASS.nunique()}개')
print('유전자 결측 셀:', missing_train)
display(train[['ID', 'SUBCLASS']].head())
if missing_train:
    raise ValueError('Train 유전자 결측이 존재합니다. 결측과 WT를 혼동하지 않도록 전처리 정책을 먼저 결정하세요.')

## 3. Train 암종 분포 탐색
**Train 자료만 사용한 기술통계**이며, 이 값을 가지고 Test 전처리를 결정하지 않는다.

In [ ]:
class_counts = train['SUBCLASS'].value_counts()
display(class_counts.rename_axis('SUBCLASS').reset_index(name='count'))
ax = class_counts.sort_values().plot.barh(figsize=(10, 9), title='Train: 암종별 환자 수')
ax.set_xlabel('환자 수'); ax.set_ylabel('SUBCLASS')
plt.tight_layout(); plt.show()

## 4. 이진화: WT=0, WT가 아닌 관찰 변이=1
문자열(예: R132H)을 보존한 원본 DataFrame은 건드리지 않는다. 현재 이진화는 **변이 유무 정보만 이용하는 기준선**이다. 결측값을 0으로 취급하지 않는다.

In [ ]:
mutation_flag = train[gene_columns].ne('WT')
# 0/1 numpy 배열 → 메모리 절약 CSR 희소 행렬
X = sparse.csr_matrix(mutation_flag.to_numpy(dtype=np.uint8))
y = train['SUBCLASS'].astype(str).to_numpy()
patient_ids = train['ID'].astype(str).to_numpy()

mutation_counts = np.asarray(X.sum(axis=1)).ravel().astype(int)
print('이진 특징 행렬:', X.shape, '| 1의 개수:', X.nnz)
print(pd.Series(mutation_counts).describe(percentiles=[.25, .5, .75, .9, .99]))
print('관측 변이 0개 환자:', int(np.sum(mutation_counts == 0)))

## 5. 같은 원본 변이 프로파일끼리 그룹화해 학습·검증 분리
Train 내 **4,384개 원본 변이 문자열**이 동일한 환자는 같은 그룹에 배치한다. 0/1 변환 결과가 우연히 같은 환자는 별개로 유지한다. `StratifiedGroupKFold` 특성상 정확한 80:20이나 클래스 비율이 보장되지는 않으므로 실제 분포도 확인한다.

**중요한 제약:** 변이 0개 환자 94명은 원본 유전자 프로파일이 완전히 동일하다. 그룹 분할을 유지하면 이들을 학습·검증에 동시에 넣을 수 없으므로 **학습에 배치**한다. 이 검증에서는 변이 0개 환자의 성능을 평가할 수 없다는 한계를 결과에 명시한다.

In [ ]:
# pandas 행별 해시를 이용해 원본 Train 유전자 프로파일 그룹화
# 해시는 매우 낮은 충돌 가능성이 있으므로 고위험 데이터에서 엄격한 동일성 보증이 필요하면 추가 확인 권장
profile_groups = pd.util.hash_pandas_object(train[gene_columns], index=False).to_numpy()
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
# 모두 WT인 94명은 동일 프로파일 그룹이다. 그 그룹이 검증에만 배치되면
# 학습 단계에 변이 0개 환자가 한 명도 없게 되므로, 학습에 포함된 첫 폴드를 선택한다.
zero_mutation_rows = (mutation_counts == 0)
fold_candidates = list(sgkf.split(X, y, groups=profile_groups))
fit_idx, valid_idx = next(
    (fi, vi) for fi, vi in fold_candidates if zero_mutation_rows[fi].any()
)
print('주의: 동일 프로파일을 분리하지 않으므로 변이 0개 환자 94명은 학습에만 존재하며,')
print('이번 검증 결과는 변이 0개 환자에 대한 일반화 성능을 평가하지 못합니다.')

assert len(set(profile_groups[fit_idx]) & set(profile_groups[valid_idx])) == 0
assert len(set(fit_idx) & set(valid_idx)) == 0
assert len(fit_idx) + len(valid_idx) == len(train)

X_fit, X_valid = X[fit_idx], X[valid_idx]
y_fit, y_valid = y[fit_idx], y[valid_idx]
print('학습:', X_fit.shape, '검증:', X_valid.shape)
print('원본 프로파일 그룹 중복:', len(set(profile_groups[fit_idx]) & set(profile_groups[valid_idx])))
display(pd.DataFrame({'fit':pd.Series(y_fit).value_counts(), 'valid':pd.Series(y_valid).value_counts()}).fillna(0).astype(int).sort_index())
assert set(np.unique(y_valid)).issubset(set(np.unique(y_fit))), '검증에만 등장하는 암종이 있습니다.' 

## 6. 변이 부담 그룹과 주요 유전자 통계
연구 가설인 *어떤 환자군에서 분류 오류가 집중되는가*를 확인하기 위한 기초 지표를 저장한다. 주요 유전자 빈도는 **학습 폴드만** 사용하여 계산한다.

In [ ]:
patient_summary = pd.DataFrame({
    'ID': patient_ids, 'SUBCLASS': y,
    'mutation_count': mutation_counts,
    'split': np.where(np.isin(np.arange(len(y)), valid_idx), 'validation', 'fit')
})
# 연구 분석용 구간. 실제 분할 임계값을 모델 튜닝에 활용할 때는 학습 폴드만 사용.
patient_summary['mutation_group'] = pd.cut(
    patient_summary['mutation_count'],
    bins=[-1, 0, 7, 27, np.inf],
    labels=['zero', 'low_1_7', 'middle_8_27', 'high_28_plus']
)
fit_gene_count = np.asarray(X_fit.sum(axis=0)).ravel().astype(int)
gene_stats = pd.DataFrame({
    'gene':gene_columns, 'mutation_count_fit':fit_gene_count,
    'mutation_rate_fit':fit_gene_count / len(fit_idx)
}).sort_values(['mutation_count_fit','gene'], ascending=[False,True])
display(gene_stats.head(15))
display(pd.crosstab(patient_summary['mutation_group'],patient_summary['split']))

## 7. 산출물 저장
추후 Baseline 모델에서는 `X_train_binary.npz`, `y_train.npy`, `split_indices.npz`를 사용하면 된다. 유전자 열 순서도 별도로 저장한다.

In [ ]:
sparse.save_npz(OUTPUT_DIR / 'X_train_binary.npz', X, compressed=True)
np.save(OUTPUT_DIR / 'y_train.npy', y)
np.save(OUTPUT_DIR / 'train_ids.npy', patient_ids)
np.savez_compressed(OUTPUT_DIR / 'split_indices.npz', fit_idx=fit_idx, valid_idx=valid_idx)
(OUTPUT_DIR / 'gene_columns.json').write_text(json.dumps(gene_columns, ensure_ascii=False, indent=2), encoding='utf-8')
patient_summary.to_csv(OUTPUT_DIR / 'train_patient_summary.csv', index=False, encoding='utf-8-sig')
class_counts.rename_axis('SUBCLASS').reset_index(name='count').to_csv(OUTPUT_DIR / 'train_class_distribution.csv', index=False, encoding='utf-8-sig')
gene_stats.to_csv(OUTPUT_DIR / 'gene_frequency_fit_only.csv', index=False, encoding='utf-8-sig')
summary = {
    'train_rows':len(train), 'genes':len(gene_columns), 'classes':int(train.SUBCLASS.nunique()),
    'non_wt_cells_train':int(X.nnz), 'fit_rows':len(fit_idx), 'validation_rows':len(valid_idx),
    'profile_overlap':0, 'train_missing':missing_train, 'seed':SEED,
    'encoding':'WT=0, observed non-WT=1; missing NOT mapped to WT',
    'test_used_for_training':False
}
(OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('저장 완료:')
for p in sorted(OUTPUT_DIR.iterdir()): print(' -', p.name)

## 8. (선택) 최종 추론 직전에만 Test 변환

**Baseline 검증 실험에서는 실행하지 말 것.** 모델 선택을 Train 교차검증으로 끝낸 후에만 사용한다. Train에서 확정된 `gene_columns` 순서로 재배열하며, Test의 통계량을 학습하지 않는다.

Test에서 결측이 발견되면 **임의로 WT(0)로 간주하지 않고 중단**한다. 결측 처리 정책을 Train 데이터에 근거해 정한 뒤 별도 구현해야 한다.

In [ ]:
# 최종 추론 단계에서만 True로 변경하세요.
RUN_TEST_TRANSFORM = False

if RUN_TEST_TRANSFORM:
    test_path = find_file(['test(2).csv', 'test(1).csv', 'test.csv'])
    test = pd.read_csv(test_path, dtype='string', low_memory=False)
    assert 'ID' in test.columns and 'SUBCLASS' not in test.columns
    assert set(test.columns) - {'ID'} == set(gene_columns), 'Train/Test 유전자 컬럼 불일치'
    assert test['ID'].is_unique and test['ID'].notna().all()
    # 값이 누락된 유전자를 WT로 간주하면 생물학적 의미가 달라지므로 중단
    missing_test = int(test[gene_columns].isna().sum().sum())
    print('Test 누락 셀:',missing_test)
    if missing_test:
        raise ValueError('Test에 결측이 있습니다. Train에 기반한 결측 처리 정책을 별도로 정의한 뒤 적용하세요.')
    X_test = sparse.csr_matrix(test[gene_columns].ne('WT').to_numpy(dtype=np.uint8))
    sparse.save_npz(OUTPUT_DIR / 'X_test_binary.npz', X_test, compressed=True)
    np.save(OUTPUT_DIR / 'test_ids.npy', test['ID'].astype(str).to_numpy())
    print('Test 변환 완료:',X_test.shape)
else:
    print('Test 미사용: Train 기반 전처리만 수행한 상태입니다.')

## 다음 단계

1. 저장된 행렬을 이용해 Baseline 모델 학습(예: Logistic Regression / ComplementNB / LightGBM)
2. **Macro F1, 암종별 Precision·Recall·F1, Confusion Matrix** 확인
3. 소수 암종 및 변이 부담 그룹별로 오분류 원인 분석
4. Train 기준으로만 전처리·특징 선택을 수정하고 동일한 검증 설계에서 비교

> **재현성 주의:** 해시 기반 그룹 분할의 seed를 고정했다. 다양한 모델의 성능을 비교할 때 동일한 분할을 유지한다.